In [ ]:
# !python -m spacy download en_core_web_sm
# !python -m spacy download de_core_news_sm

In [ ]:
!pip install datasets
!pip install sentencepiece
!pip install evaluate
!pip install bert_score
!pip install accelerate -U
!pip install sacrebleu

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 542.0/542.0 kB 3.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 11.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.1/194.1 kB 10.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 kB 12.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 388.9/388.9 kB 24.7 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface-hub 0.20.3
    Uninstalling huggingface-hub-0.20.3:
      Successfully uninstalled huggingface-hub-0.20.3
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 1.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 953.5 kB/s eta 0:00:00
  Using cached nvidia_cuda_nvrtc_cu12-12.1.105-py3-none-manylinux1_x86_64.whl (23.7 MB)
  Using cached nvidia_cuda_runtime_cu12-12.1.105-py3-none-manylinux1_x86_64.whl (823 kB)
  Using cached nvidia_cuda_cupti_cu12-12.1.105-py3-none-

In [ ]:
import os
import torch
import datasets
import spacy
import numpy as np
from evaluate import load
from torch.utils.data import Dataset
from transformers import T5ForConditionalGeneration, T5Tokenizer

In [ ]:
import pandas as pd

In [ ]:
from google.colab import drive
drive.mount('/content/drive/')

Mounted at /content/drive/


In [ ]:
path = "/content/drive/MyDrive/NLP_2024/Project/Data_after_cleaning_and_splits/"
# os.chdir(path)

In [ ]:
# spacy.load('en_core_web_sm')
# spacy.load('de_core_news_sm')

In [ ]:
train_dataset = pd.read_csv(path+'train_dataset_final.csv')
test_dataset = pd.read_csv(path+'test_dataset_final.csv')
val_dataset = pd.read_csv(path+'val_dataset_final.csv')
train_dataset = train_dataset[train_dataset['offensiveYN']==1.0]
test_dataset = test_dataset[test_dataset['offensiveYN']==1.0]
val_dataset = val_dataset[val_dataset['offensiveYN']==1.0]

In [ ]:
print(len(train_dataset))
print(len(test_dataset))
print(len(val_dataset))

10974
1750
1620


In [ ]:
train_dataset = train_dataset[['post','targetMinority','targetCategory','rationale','targetStereotype']]
test_dataset = test_dataset[['post','targetMinority','targetCategory','rationale','targetStereotype']]
val_dataset = val_dataset[['post','targetMinority','targetCategory','rationale','targetStereotype']]
train_dataset = train_dataset.dropna()
test_dataset = test_dataset.dropna()
val_dataset = val_dataset.dropna()

In [ ]:
print(len(train_dataset))
print(len(test_dataset))
print(len(val_dataset))

10974
1750
1620


In [ ]:
test_dataset

,post,targetMinority,targetCategory,rationale,targetStereotype
1,When u hitting it from the back u gotta call h...,women,gender,This post promotes the use of degrading langua...,sexually promiscuous women are called sluts
3,You niggas talking to these females trying to ...,women,gender,This post displays a clear disdain and contemp...,women are self-centered/egotistical
5,A Nigguh spitting dat crack to your bitch,black folks,race,This post is considered hateful because it pro...,black people are crackheads
6,RT @lesxo_: cant turn a hoe into a husband either,"women, trans men",gender,This post is considered hateful because it imp...,women who have sex before marriage are ruined.
7,"""@ReachtheeCLIMAX: bitches can't cook but know...",women,gender,This post is hateful because it objectifies an...,women are used for sex
...,...,...,...,...,...
3784,Female depression and Jon Snow have one thing ...,women,gender,This post is hateful because it is making an i...,women can’t be depressed
3786,A white male toddler needs to have its head sm...,"black folks, latino/latina folks",race,This post promotes violence and hate towards a...,blacks are niggers
3787,women are gay\nwomen are all lesbians and that...,"women, lesbian women",gender,This post is hateful because it displays a cle...,women hate men
3796,If you became chad tier good looking through s...,women,gender,"This post shows a clear desire for revenge, wh...",deserve revenge for rejecting men


In [ ]:
# Decode translations
# test_translations = [tokenizer.decode(output, skip_special_tokens=True) for output in outputs]

In [ ]:
test_dataset["targetMinority"].value_counts()

targetMinority
black folks                                      532
women                                            302
jewish folks                                     196
muslim folks                                      80
physically disabled folks                         43
                                                ... 
assault victims, sexual abuse, incest victims      1
chinese people                                     1
muslim folks, arabic folks                         1
korean people                                      1
asian folks, Indians                               1
Name: count, Length: 189, dtype: int64

In [ ]:
test_dataset['targetMinority'].nunique()

189

In [ ]:
test_dataset['targetMinority'].unique()

array(['women', 'black folks', 'women, trans men', 'gay men',
       'latino/latina folks', 'lesbian women, gay men',
       'overweight/fat folks', 'asian folks',
       'folks with mental illness/disorder, mentally disabled folks',
       'muslim folks', 'gay men, heterosexuals', 'men',
       'mentally disabled folks', 'jewish folks', 'liberals',
       'white people', 'assault victims', 'immigrants', 'arabic folks',
       'Colombians', 'christian folks', 'dominican',
       'physically disabled folks', 'folks with mental illness/disorder',
       'conservatives', 'cops', 'labour party folks', 'terrorism victims',
       'United Kingdom (U.K.)', 'soviets', 'White people', 'Russians',
       'lesbian women', 'Pagan', 'muslim folks, immigrants',
       'korean people', 'Afghan people', 'priests', 'Native Americans',
       'white folks', 'Thai', 'trans women', 'Syrians', 'indian folks',
       'child sexual assault victims', 'Ethiopia', 'ethiopens',
       'poor folks', 'India', 'Chi

In [ ]:
from transformers import AutoTokenizer

t5_checkpoint = "google-t5/t5-small"
finetune_tokenizer = AutoTokenizer.from_pretrained(t5_checkpoint)

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/2.32k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

In [ ]:
task_prefix = "What is the implication of this hateful post:"
# task_mid1 = " with target minority "
# task_mid2 = " with target category "
task_mid3 = "\nLet's think about this step by step. "
final_answer_prompt = " Hence, the implication is: "
# post targetMinority targetCategory rationale
# targetStereotype

In [ ]:
class CustomDataset(Dataset):

  def __init__(self, data,tokenizer):
    # col names ['post','targetMinority','targetCategory','rationale','targetStereotype']
    self.posts = data['post'].tolist()
    self.minority = data['targetMinority'].tolist()
    self.category = data['targetCategory'].tolist()
    self.rationale = data['rationale'].tolist()
    self.label = data['targetStereotype'].tolist()
    self.tokenizer = tokenizer

  def __len__(self):
    return len(self.posts)

  def __getitem__(self, idx):
    inputs = task_prefix + str(self.posts[idx]) + "?" +task_mid3+ str(self.rationale[idx]) + final_answer_prompt
    targets = str(self.label[idx])
    input_enc = self.tokenizer(inputs, text_target=targets, max_length=512, padding=True ,truncation=True)

    # print(input_enc)
    return {
          'input_ids': input_enc['input_ids'],
          'attention_mask': input_enc['attention_mask'],
          'labels':input_enc['labels'],
          'input_text': inputs,
          'target_text': targets
          }

In [ ]:
# tokenized_dataset_train = train_dataset.map(preprocess_function, batched=True)
tokenized_dataset_train = CustomDataset(train_dataset,finetune_tokenizer)

In [ ]:
tokenized_dataset_train[500]

{'input_ids': [363,
  19,
  8,
  3,
  28722,
  13,
  48,
  5591,
  1329,
  442,
  10,
  5934,
  3320,
  63,
  425,
  226,
  102,
  52,
  3772,
  10,
  24500,
  3,
  63,
  32,
  720,
  524,
  114,
  10,
  2649,
  1303,
  17,
  5,
  509,
  87,
  1265,
  20611,
  15382,
  1298,
  102,
  7,
  210,
  58,
  1563,
  31,
  7,
  317,
  81,
  48,
  1147,
  57,
  1147,
  5,
  100,
  442,
  19,
  1702,
  5591,
  1329,
  250,
  34,
  2519,
  7,
  3,
  9,
  1817,
  32,
  122,
  63,
  29,
  3040,
  11,
  3735,
  8587,
  7525,
  1587,
  887,
  5,
  37,
  169,
  13,
  20,
  3822,
  6546,
  1612,
  13308,
  115,
  7059,
  8512,
  11,
  8,
  3735,
  2420,
  13,
  887,
  38,
  12143,
  4820,
  12,
  36,
  15319,
  28,
  3,
  9,
  824,
  7525,
  31324,
  6203,
  10947,
  7285,
  26524,
  7,
  11,
  2519,
  7,
  3,
  9,
  12068,
  903,
  13,
  3079,
  5,
  100,
  686,
  13,
  1612,
  11,
  7525,
  54,
  4139,
  12,
  3,
  9,
  1543,
  13,
  31973,
  11,
  4756,
  1587,
  887,
  5,
  5433,
  6,
  8,
  169,
 

In [ ]:
c = 0
for i in tokenized_dataset_train:
  print(i)
  print(len(i['input_ids']))
  if c>2:
    break
  c+=1
# just using to see length of input ids

{'input_ids': [363, 19, 8, 3, 28722, 13, 48, 5591, 1329, 442, 10, 5934, 3320, 23, 2703, 956, 32, 10, 3, 13076, 12417, 3, 89, 4636, 53, 3, 7315, 1304, 312, 22780, 29, 5, 148, 8882, 5341, 13721, 19126, 2293, 29, 63, 21622, 3, 89, 9, 122, 10779, 5, 58, 1563, 31, 7, 317, 81, 48, 1147, 57, 1147, 5, 100, 442, 19, 5591, 1329, 250, 34, 2284, 20, 3822, 6546, 1612, 11, 3, 7, 40, 3589, 12, 3211, 312, 22780, 29, 2549, 6, 3, 9, 8304, 1001, 17893, 5, 37, 169, 13, 1353, 224, 38, 96, 7315, 1304, 976, 96, 22498, 109, 2293, 29, 63, 976, 11, 96, 2157, 4397, 121, 33, 7447, 12130, 11, 20, 12450, 2610, 1587, 1001, 151, 5, 5433, 6, 8, 442, 92, 2284, 8, 1448, 96, 89, 9, 122, 10779, 121, 38, 46, 21548, 6, 31324, 1014, 13503, 27426, 5, 37, 1879, 5739, 13, 8, 442, 19, 8299, 11, 3353, 28, 11213, 6, 2924, 3, 9, 964, 8762, 12, 6263, 11, 36, 26192, 8, 568, 2799, 5, 100, 686, 13, 1612, 11, 13972, 13, 3, 9, 806, 1964, 11, 6949, 12602, 19, 29452, 11, 2519, 7, 5591, 11, 9192, 257, 5, 3, 13151, 6, 8, 3, 28722, 19, 10, 1]

In [ ]:
# tokenized_dataset_val = val_dataset.map(preprocess_function, batched=True)
tokenized_dataset_val = CustomDataset(val_dataset,finetune_tokenizer)

In [ ]:
from transformers import DataCollatorForSeq2Seq

data_collator = DataCollatorForSeq2Seq(tokenizer=finetune_tokenizer, model=t5_checkpoint)

In [ ]:
metric = load("sacrebleu")

In [ ]:
from transformers import AutoModelForSeq2SeqLM, Seq2SeqTrainingArguments, Seq2SeqTrainer

finetune_model = AutoModelForSeq2SeqLM.from_pretrained(t5_checkpoint)

config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/242M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [ ]:
training_args = Seq2SeqTrainingArguments(
    output_dir="T5_cot_finetuned",
    evaluation_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    weight_decay=0.01,
    save_total_limit=3,
    num_train_epochs=30,
    predict_with_generate=True
)

trainer = Seq2SeqTrainer(
    model=finetune_model,
    args=training_args,
    train_dataset=tokenized_dataset_train,
    eval_dataset=tokenized_dataset_val,
    tokenizer=finetune_tokenizer,
    data_collator=data_collator,
)

In [ ]:
train_results = trainer.train()

Epoch,Training Loss,Validation Loss
1,2.581100,2.283224
2,2.312400,2.140505
3,2.190200,2.070146
4,2.114900,2.021649
5,2.054500,1.982060
6,2.006000,1.955533
7,1.998700,1.935576
8,1.923400,1.907605
9,1.920100,1.902766
10,1.862600,1.884634


In [ ]:
import pickle

model_path = "/content/drive/MyDrive/NLP_2024/Project/Generation/T5_COT_Generation.pkl"

with open(model_path, 'wb') as f:
    pickle.dump(finetune_model, f)

In [ ]:
# finetune_tokenizer.save_pretrained("")

In [ ]:
!pip install rouge-score

  Preparing metadata (setup.py) ... done
  Created wheel for rouge-score: filename=rouge_score-0.1.2-py3-none-any.whl size=24933 sha256=5181586aa59a82f2b485247a290115e458ba3214945e5f16205cef535852b7d4
  Stored in directory: /root/.cache/pip/wheels/5f/dd/89/461065a73be61a532ff8599a28e9beef17985c9e9c31e541b4
Successfully built rouge-score


In [ ]:
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
# finetune_model = AutoModelForSeq2SeqLM.from_pretrained('finetuned_T5_final')

In [ ]:
# finetune_model.to(DEVICE)

In [ ]:
test_posts = test_dataset['post'].tolist()
test_minority = test_dataset['targetMinority'].tolist()
test_category = test_dataset['targetCategory'].tolist()
test_rationale = test_dataset['rationale'].tolist()
test_label = test_dataset['targetStereotype'].tolist()

In [ ]:
test_targets = []
test_generated = []
i = 0
for idx in range(len(test_posts)):
    inputs = task_prefix + str(test_posts[idx]) + "?" +task_mid3+ str(test_rationale[idx]) + final_answer_prompt
    input_ids = finetune_tokenizer.encode(inputs, return_tensors="pt").to(DEVICE)
    output = finetune_model.generate(input_ids, max_length=50, num_beams=4, early_stopping=True)
    output_text = finetune_tokenizer.decode(output[0], skip_special_tokens=True)
    test_generated.append(output_text)
    test_targets.append(test_label[idx])

In [ ]:
count = 20
i = 0
for j in range(len(test_generated)):
  print("Target:", test_targets[j])
  print("Generated:", test_generated[j])
  print("**********************************************")

Streaming output truncated to the last 5000 lines.
Generated: women are bitches
**********************************************
Target: gay men are indecisive
Generated: black people are faggots
**********************************************
Target: gay men are ugly
Generated: gay men are faggots
**********************************************
Target: women don't value the right things
Generated: women are hoes
**********************************************
Target: women are promiscuous
Generated: black people are hoes
**********************************************
Target: fat women are unattractive
Generated: black people are fat
**********************************************
Target: are all bitches
Generated: women are bitches
**********************************************
Target: act like children
Generated: mexicans are bitches
**********************************************
Target: women are hoes
Generated: black people are hoes
**********************************************
Target: 

In [ ]:
!pip install rouge-score

In [ ]:
import evaluate
rouge = evaluate.load('rouge')
results = rouge.compute(predictions=test_generated, references=test_targets)
print(results)

{'rouge1': 0.31308437118437105, 'rouge2': 0.16985774225774225, 'rougeL': 0.3109927154544804, 'rougeLsum': 0.3113439982986203}


In [ ]:
from datasets import load_metric
metric = load_metric("rouge")
results = metric.compute(predictions=test_generated, references=test_targets)
# print(results)
result = list(results.items())
for index in range(len(result)-1):
  type_, scores = result[index]
  precision = scores.mid.precision
  recall = scores.mid.recall
  f1 = scores.mid.fmeasure
  print(str(type_)+" with precision "+str(precision)+" with recall "+str(recall)+" and F1 score "+str(f1))

<ipython-input-36-4a703b08e2ee>:2: FutureWarning: load_metric is deprecated and will be removed in the next major version of datasets. Use 'evaluate.load' instead, from the new library 🤗 Evaluate: https://huggingface.co/docs/evaluate
  metric = load_metric("rouge")
/usr/local/lib/python3.10/dist-packages/datasets/load.py:759: FutureWarning: The repository for rouge contains custom code which must be executed to correctly load the metric. You can inspect the repository content at https://raw.githubusercontent.com/huggingface/datasets/2.19.0/metrics/rouge/rouge.py
You can avoid this message in future by passing the argument `trust_remote_code=True`.
Passing `trust_remote_code=True` will be mandatory to load this metric from the next major release of `datasets`.
  warnings.warn(


rouge1 with precision 0.3300020408163262 with recall 0.31107646391890054 and F1 score 0.31308437118437105
rouge2 with precision 0.17613027210884355 with recall 0.17084206349206327 and F1 score 0.16985774225774225
rougeL with precision 0.32769489795918316 with recall 0.30911819678920494 and F1 score 0.3109927154544804


In [ ]:
from bert_score import score as bert_score
Ptestft, Rtestft, F1testft = bert_score(cands=test_generated, refs=test_targets, lang='en', verbose=True)
print("BERTScore Precision:", Ptestft.mean().item())
print("BERTScore Recall:", Rtestft.mean().item())
print("BERTScore F1:", F1testft.mean().item())

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

...

calculating scores...
computing bert embedding.


  0%|          | 0/30 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/28 [00:00<?, ?it/s]

done in 4.80 seconds, 364.52 sentences/sec
BERTScore Precision: 0.8898860216140747
BERTScore Recall: 0.8925727605819702
BERTScore F1: 0.8910824656486511


In [ ]:
import nltk
from nltk.translate.bleu_score import corpus_bleu

# Example target and generated sentences
# target_sentences = ['The cat is on the mat', 'There is a dog', 'The sky is blue']
# generated_sentences = ['The cat is sitting on the mat', 'A dog is here', 'The sky is blue']

# Tokenize sentences
target_sentences_tokenized = [sentence.split() for sentence in test_targets]
generated_sentences_tokenized = [sentence.split() for sentence in test_generated]

# Calculate BLEU score
bleu_score = corpus_bleu([[ref] for ref in target_sentences_tokenized], generated_sentences_tokenized)

print("BLEU Score:", bleu_score)

BLEU Score: 0.1018801502678393
